In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow_addons.optimizers import AdamW
import keras
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Conv2D, BatchNormalization, MaxPooling2D
from tensorflow.keras.layers import Input, ZeroPadding2D, Add
from keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
from tensorflow.keras.metrics import Precision, AUC, Recall
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers.legacy import SGD, Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LeakyReLU
import os

In [ ]:
from keras.preprocessing.image import ImageDataGenerator

BATCH_SIZE=32

SEED=1345

train_datagen=ImageDataGenerator(rescale=1./255,
                                shear_range=0,
                                zoom_range=0.2, rotation_range = 30)

validation_datagen=ImageDataGenerator(rescale=1./255)
test_datagen=ImageDataGenerator(rescale=1./255)


#Defining directories for train,validation,test 
train_dir = 'Splitted2D-19/train'
validation_dir = 'Splitted2D-19/val'
test_dir = 'Splitted2D-19/test'


#Defining generatores for train,validation,test

train_generator=train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    shuffle=True,
    seed = SEED,
    batch_size= 32,
    class_mode ='categorical',
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(224, 224),
    seed = SEED,
    shuffle=True,
    batch_size= 32,
    class_mode ='categorical',
)

# Define generator for test set using flow_from_directory
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    shuffle=True,
    seed = SEED,
    batch_size = 32,
    class_mode ='categorical',
)

In [ ]:
input_layer = Input(shape = (224, 224, 3))
conv1_pad = ZeroPadding2D(3)(input_layer)

conv_conv1 = Conv2D(64, kernel_size = (7, 7), strides = (2, 2), use_bias = False, 
                    name = 'conv_conv1', kernel_initializer = 'he_normal')(conv1_pad)
conv_bn1 = BatchNormalization(name = 'conv_bn1')(conv_conv1)
conv_relu1 = LeakyReLU(0.1, name = 'conv_relu1')(conv_bn1)
conv_poolpad1 = ZeroPadding2D(1, name = 'conv_poolpad1')(conv_relu1)
conv_pool1 = MaxPooling2D(pool_size = (3, 3), strides = (2, 2), name = 'conv_pool1')(conv_poolpad1)


'''block 1'''
b1_conv1 = Conv2D(64, kernel_size = (3, 3), strides = (1, 1), use_bias = False, padding = "same", 
                   name = 'b1_conv1', kernel_initializer = 'he_normal')(conv_pool1)
b1_bn1 = BatchNormalization(name = 'b1_bn1')(b1_conv1)
b1_relu1 = LeakyReLU(0.1, name = 'b1_relu1')(b1_bn1)

b1_0_conv2 = Conv2D(256, kernel_size = (3, 3), strides = (1, 1), use_bias = False, padding = "same",
                  name = 'b1_0_conv2', kernel_initializer = 'he_normal')(conv_pool1)
b1_1_conv3 = Conv2D(256, kernel_size = (3, 3), strides = (1, 1), use_bias = False, padding = "same", 
                  name = 'b1_1_conv3', kernel_initializer = 'he_normal')(b1_relu1)
b1_0_bn2 = BatchNormalization(name = 'b1_0_bn2')(b1_0_conv2)
b1_1_bn3 = BatchNormalization(name = 'b1_1_bn3')(b1_1_conv3)
b1_add = Add(name = 'b1_add')([b1_0_bn2, b1_1_bn3])
b1_out = LeakyReLU(0.1, name = 'b1_relu_out')(b1_add)

'''block 2'''
b2_conv1 = Conv2D(64, kernel_size = (3, 3), strides = (1, 1), use_bias = False, padding = "same", 
                  name = 'b2_conv1', kernel_initializer = 'he_normal')(b1_out)
b2_bn1 = BatchNormalization(name = 'b2_bn1')(b2_conv1)
b2_relu1 = LeakyReLU(0.1, name = 'b2_relu1')(b2_bn1)
b2_conv2 = Conv2D(64, kernel_size = (3, 3), strides = (1, 1), use_bias = False, padding = "same",
                  name = 'b2_conv2', kernel_initializer = 'he_normal')(b2_relu1)
b2_bn2 = BatchNormalization(name = 'b2_bn2')(b2_conv2)
b2_relu2 = LeakyReLU(0.1, name = 'b2_relu2')(b2_bn2)
b2_conv3 = Conv2D(256, kernel_size = (3, 3), strides = (1, 1), use_bias = False, padding = "same",
                  name = 'b2_conv3', kernel_initializer = 'he_normal')(b2_bn2)
b2_bn3 = BatchNormalization(name = 'b2_bn3')(b2_conv3)
b2_add = Add()([b1_out, b2_bn3])
b2_out = LeakyReLU(0.1, name = 'b2_relu_out')(b2_add)

'''block 3'''
b3_conv1 = Conv2D(64, kernel_size = (3, 3), strides = (1, 1), use_bias = False, padding = "same", 
                  name = 'b3_conv1', kernel_initializer = 'he_normal')(b2_out)
b3_bn1 = BatchNormalization(name = 'b3_bn1')(b3_conv1)
b3_relu1 = LeakyReLU(0.1, name = 'b3_relu1')(b3_bn1)
b3_conv2 = Conv2D(64, kernel_size = (3, 3), strides = (1, 1), use_bias = False, padding = "same", 
                  name = 'b3_conv2', kernel_initializer = 'he_normal')(b3_relu1)
b3_bn2 = BatchNormalization(name = 'b3_bn2')(b3_conv2)
b3_relu2 = LeakyReLU(0.1, name = 'b3_relu2')(b3_bn2)
b3_conv3 = Conv2D(256, kernel_size = (3, 3), strides = (1, 1), use_bias = False, padding = "same", 
                  name = 'b3_conv3', kernel_initializer = 'he_normal')(b3_relu2)
b3_bn3 = BatchNormalization(name = 'b3_bn3')(b3_conv3)
b3_add = Add()([b2_out, b3_bn3])
b3_out = LeakyReLU(0.1, name = 'b3_relu_out')(b3_add)

conv_blocks = Model(input_layer, b3_out)

In [ ]:
global_pooling = GlobalAveragePooling2D()
batch_norm = BatchNormalization()
dense_layers = Sequential([
    Dropout(0.15),
    Dense(1024, activation = LeakyReLU(0.1), kernel_initializer = 'he_normal'),
    Dropout(0.15),
    Dense(512, activation = LeakyReLU(0.1), kernel_initializer = 'he_normal'),
    Dropout(0.15), 
    Dense(256, activation = LeakyReLU(0.1), kernel_initializer = 'he_normal'),
    Dropout(0.15)
])
prediction_layer = Dense(4, activation = 'softmax', kernel_initializer = 'glorot_normal')
model = Sequential([
    conv_blocks,
    global_pooling,
    batch_norm,
    dense_layers,
    prediction_layer
])

model.compile(optimizer = Adam(learning_rate = 0.005, clipvalue=1.0), loss = 'categorical_crossentropy', 
              metrics=['accuracy',tf.keras.metrics.AUC(),
                        tf.keras.metrics.Precision(),
                        tf.keras.metrics.Recall(),])

In [ ]:
model.summary()

In [ ]:
from tensorflow.keras.initializers import HeNormal
conv_layers = Sequential([
    Conv2D(64, kernel_size = (3, 3), strides = (1, 1), padding = "same", input_shape = (224, 224, 3), 
           kernel_initializer = HeNormal(), activation = LeakyReLU(0.3)),
    Conv2D(64, kernel_size = (3, 3), strides = (1, 1), padding = "same", kernel_initializer = HeNormal(),
          activation = LeakyReLU(0.3)),
    MaxPooling2D(pool_size = (3, 3), strides = (2, 2), padding = "same"),
    Conv2D(128, kernel_size = (3, 3), strides = (1, 1), padding = "same", kernel_initializer = HeNormal(),
          activation = LeakyReLU(0.3)),
    Conv2D(128, kernel_size = (3, 3), strides = (1, 1), padding = "same", kernel_initializer = HeNormal(),
          activation = LeakyReLU(0.1)),
    MaxPooling2D(pool_size = (3, 3), strides = (2, 2), padding = "same"),
    BatchNormalization(),
    Conv2D(256, kernel_size = (3, 3), strides = (1, 1), padding = "same", kernel_initializer = HeNormal(),
          activation = LeakyReLU(0.3)),
    Conv2D(256, kernel_size = (3, 3), strides = (1, 1), padding = "same", kernel_initializer = HeNormal(),
          activation = LeakyReLU(0.3)),
    MaxPooling2D(pool_size = (3, 3), strides = (2, 2), padding = "same"),
    Conv2D(512, kernel_size = (3, 3), strides = (1, 1), padding = "same", kernel_initializer = HeNormal(),
          activation = LeakyReLU(0.3)),
    Conv2D(512, kernel_size = (3, 3), strides = (1, 1), padding = "same", kernel_initializer = HeNormal(),
          activation = LeakyReLU(0.3)),
    MaxPooling2D(pool_size = (3, 3), strides = (2, 2), padding = "same")
])
global_pooling = GlobalAveragePooling2D()
dense_layer = Sequential([
    BatchNormalization(),
    Dense(1024, activation = LeakyReLU(0.3), kernel_initializer = HeNormal()),
    Dropout(0.2),
    Dense(512, activation = LeakyReLU(0.3), kernel_initializer = HeNormal()),
    Dropout(0.2),
    Dense(256, activation = LeakyReLU(0.3), kernel_initializer = HeNormal()),
    Dropout(0.2)
])
prediction_layer = Dense(4, activation = 'softmax', kernel_initializer = 'glorot_uniform')

model = Sequential([
    conv_layers,
    global_pooling,
    dense_layer,
    prediction_layer
])

model.compile(optimizer = Adam(learning_rate = 0.005, clipvalue=1.0), loss = 'categorical_crossentropy', 
              metrics=['accuracy',tf.keras.metrics.AUC(),
                        tf.keras.metrics.Precision(),
                        tf.keras.metrics.Recall(),])

In [ ]:
model.summary()

In [ ]:

history=model.fit(train_generator,
                        validation_data=validation_generator,
                        steps_per_epoch=len(train_generator),
                        epochs = 100,
                        verbose = 1)